<a href="https://colab.research.google.com/github/KBCoronado/MachineLearning/blob/main/Practica_2_Regresi%C3%B3n_Log%C3%ADstica.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go

# Importamos librerías necesarias para regresión logística y evaluación de modelos
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, precision_recall_curve

prestamos_df = pd.read_csv('lending_club_2007_2011_6_states (2).csv')

prestamos_df.head()

,loan_amnt,funded_amnt,funded_amnt_inv,term,int_rate,installment,grade,sub_grade,emp_title,emp_length,...,application_type,acc_now_delinq,chargeoff_within_12_mths,delinq_amnt,pub_rec_bankruptcies,tax_liens,hardship_flag,disbursement_method,debt_settlement_flag,debt_settlement_flag_date
0,2400,2400,2400.0,36 months,15.96,84.33,C,C5,NaN,10+ years,...,Individual,0.0,0.0,0.0,0.0,0.0,N,Cash,N,NaN
1,10000,10000,10000.0,36 months,13.49,339.31,C,C1,AIR RESOURCES BOARD,10+ years,...,Individual,0.0,0.0,0.0,0.0,0.0,N,Cash,N,NaN
2,3000,3000,3000.0,36 months,18.64,109.43,E,E1,MKC Accounting,9 years,...,Individual,0.0,0.0,0.0,0.0,0.0,N,Cash,N,NaN
3,5600,5600,5600.0,60 months,21.28,152.39,F,F2,NaN,4 years,...,Individual,0.0,0.0,0.0,0.0,0.0,N,Cash,N,NaN
4,5375,5375,5350.0,60 months,12.69,121.45,B,B5,Starbucks,< 1 year,...,Individual,0.0,0.0,0.0,0.0,0.0,N,Cash,N,NaN


In [3]:
prestamos_df["grade_code"] = prestamos_df["grade"].astype("category").cat.codes
prestamos_df["purpose_code"] = prestamos_df["purpose"].astype("category").cat.codes
prestamos_df["addr_state_code"] = prestamos_df["addr_state"].astype("category").cat.codes
prestamos_df["home_ownership_code"] = prestamos_df["home_ownership"].astype("category").cat.codes
prestamos_df["loan_term_year"] = prestamos_df["term"].str.extract('(\d+)').astype(int)

prestamos_df["int_rate_clean"] = prestamos_df["int_rate"]

prestamos_df["installment_clean"] = prestamos_df["installment"]

prestamos_df["repaid"] = prestamos_df["loan_status"].apply(
    lambda x: 1 if x == "Fully Paid" else 0
)

<>:5: SyntaxWarning: invalid escape sequence '\d'
<>:5: SyntaxWarning: invalid escape sequence '\d'
/tmp/ipykernel_751/1456439568.py:5: SyntaxWarning: invalid escape sequence '\d'
  prestamos_df["loan_term_year"] = prestamos_df["term"].str.extract('(\d+)').astype(int)


In [4]:
prestamos_df.head()

,loan_amnt,funded_amnt,funded_amnt_inv,term,int_rate,installment,grade,sub_grade,emp_title,emp_length,...,debt_settlement_flag,debt_settlement_flag_date,grade_code,purpose_code,addr_state_code,home_ownership_code,loan_term_year,int_rate_clean,installment_clean,repaid
0,2400,2400,2400.0,36 months,15.96,84.33,C,C5,NaN,10+ years,...,N,NaN,2,11,2,4,36,15.96,84.33,1
1,10000,10000,10000.0,36 months,13.49,339.31,C,C1,AIR RESOURCES BOARD,10+ years,...,N,NaN,2,9,0,4,36,13.49,339.31,1
2,3000,3000,3000.0,36 months,18.64,109.43,E,E1,MKC Accounting,9 years,...,N,NaN,4,0,0,4,36,18.64,109.43,1
3,5600,5600,5600.0,60 months,21.28,152.39,F,F2,NaN,4 years,...,N,NaN,5,11,0,3,60,21.28,152.39,0
4,5375,5375,5350.0,60 months,12.69,121.45,B,B5,Starbucks,< 1 year,...,N,NaN,1,9,5,4,60,12.69,121.45,0


In [5]:
X = prestamos_df[['funded_amnt', "int_rate", "grade_code",
                  'purpose_code', 'addr_state_code',
                  'home_ownership_code', 'annual_inc',
                  'dti', 'revol_util',
                  'pub_rec_bankruptcies']]

X = X.fillna(0)

y = prestamos_df["repaid"]

In [6]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.4, random_state=42)

In [7]:
X_train.shape, X_test.shape, y_train.shape, y_test.shape

((11944, 10), (7964, 10), (11944,), (7964,))

In [8]:
clf1 = LogisticRegression(random_state=0)
clf1.fit(X_train, y_train)

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


LogisticRegression(random_state=0)

In [9]:
print("Precision del clasificador en fase de entrenamiento",
      clf1.score(X_train, y_train))

Precision del clasificador en fase de entrenamiento 0.8551574012056262


In [10]:
y_pred = clf1.predict(X_test)
print("\nReporte de métricas del clasificador: \n",
      classification_report(y_test, y_pred, target_names=["No Pagado", "Pagado"]))
print( "Precisión:", clf1.score(X_test, y_test) )


Reporte de métricas del clasificador: 
               precision    recall  f1-score   support

   No Pagado       0.00      0.00      0.00      1220
      Pagado       0.85      1.00      0.92      6744

    accuracy                           0.85      7964
   macro avg       0.42      0.50      0.46      7964
weighted avg       0.72      0.85      0.78      7964

Precisión: 0.8463083877448518


In [11]:
matriz_sin_balanceo = confusion_matrix(y_test, y_pred )
print(f'Matriz Confusion:\n', matriz_sin_balanceo)

Matriz Confusion:
 [[   0 1220]
 [   4 6740]]


In [12]:
clf2 = LogisticRegression(random_state=0, class_weight="balanced")
clf2.fit(X_train, y_train)

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


LogisticRegression(class_weight='balanced', random_state=0)

In [13]:
print("Exactitud del modelo de entrenamiento:", clf2.score(X_train, y_train))

Exactitud del modelo de entrenamiento: 0.5429504353650368


In [14]:
y_pred2 = clf2.predict(X_test)
print("\nReporte de métricas del clasificador: \n",
      classification_report(y_test, y_pred2, target_names=["No Pagado", "Pagado"]))
print("Precisión:", clf2.score(X_test, y_test) )


Reporte de métricas del clasificador: 
               precision    recall  f1-score   support

   No Pagado       0.20      0.70      0.31      1220
      Pagado       0.90      0.50      0.65      6744

    accuracy                           0.53      7964
   macro avg       0.55      0.60      0.48      7964
weighted avg       0.80      0.53      0.60      7964

Precisión: 0.5334003013561025


In [15]:
matriz_balanceo = confusion_matrix(y_test, y_pred2)
print(f'Matriz Confusion:\n', matriz_balanceo)

Matriz Confusion:
 [[ 853  367]
 [3349 3395]]


In [16]:
from sklearn.inspection import permutation_importance
result = permutation_importance(clf2, X_test, y_test, n_repeats=30, random_state=42)
importancia = pd.DataFrame({
    "Variable": X.columns,
    "Importancia Media": result.importances_mean,
    "Desviación": result.importances_std
}).sort_values("Importancia Media", ascending=False)
print(importancia)

               Variable  Importancia Media  Desviación
1              int_rate           0.037929    0.003866
6            annual_inc           0.025950    0.003463
2            grade_code           0.012330    0.002339
7                   dti           0.005069    0.002838
0           funded_amnt           0.003846    0.001790
5   home_ownership_code           0.003399    0.001374
3          purpose_code           0.002829    0.001831
4       addr_state_code           0.000703    0.001673
8            revol_util           0.000465    0.000667
9  pub_rec_bankruptcies           0.000025    0.000050


In [17]:
import plotly.express as px
fig4 = px.bar(
    importancia,
    x="Variable",
    y="Importancia Media",
    error_y="Desviación",
    title="Importancia de características (Regresión Logística - Permutation Importance)",
    text_auto=".3f",
    color="Variable"
)
fig4.update_layout(width=800, height=600)
fig4.show()

In [18]:
from sklearn.preprocessing import StandardScaler
X = prestamos_df[['funded_amnt', "int_rate", "grade_code", 'purpose_code', 'addr_state_code',
                  'home_ownership_code', 'annual_inc', 'dti', 'revol_util',
                  'pub_rec_bankruptcies']]
y = prestamos_df["repaid"]
X = X.fillna(0)
X_train, X_test, y_train, y_test = train_test_split(X, y,test_size=0.4,random_state=42,stratify=y)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
RL_scaled = LogisticRegression(random_state=0, class_weight='balanced')
RL_scaled.fit(X_train_scaled, y_train)
y_pred_scaled = RL_scaled.predict(X_test_scaled)
print("\nReporte de clasificación:")
print(classification_report(y_test, y_pred_scaled, target_names=["No Pagado","Pagado"]))


Reporte de clasificación:
              precision    recall  f1-score   support

   No Pagado       0.22      0.61      0.32      1177
      Pagado       0.90      0.62      0.73      6787

    accuracy                           0.61      7964
   macro avg       0.56      0.61      0.52      7964
weighted avg       0.80      0.61      0.67      7964



In [19]:
print("\nMatriz de Confusión:")
print(confusion_matrix(y_test, y_pred_scaled))


Matriz de Confusión:
[[ 718  459]
 [2611 4176]]


In [20]:
from sklearn.inspection import permutation_importance
result = permutation_importance(RL_scaled, X_test_scaled, y_test, n_repeats=30, random_state=42)
importancia = pd.DataFrame({
    "Variable": X.columns,
    "Importancia Media": result.importances_mean,
    "Desviación": result.importances_std
}).sort_values("Importancia Media", ascending=False)
print(importancia)

               Variable  Importancia Media  Desviación
1              int_rate           0.039176    0.004599
6            annual_inc           0.008970    0.002704
5   home_ownership_code           0.002578    0.001297
3          purpose_code           0.001637    0.002298
2            grade_code           0.000159    0.000279
7                   dti          -0.000306    0.000761
4       addr_state_code          -0.000306    0.002118
0           funded_amnt          -0.001406    0.002029
9  pub_rec_bankruptcies          -0.002047    0.000922
8            revol_util          -0.002474    0.001081


In [21]:
import plotly.express as px
fig4 = px.bar(importancia, x="Variable", y="Importancia Media",
              error_y="Desviación", title="Importancia de características (Regresión Logística - Per",
              text_auto=".3f", color="Variable")
fig4.update_layout(width=800, height=600)
fig4.show()

In [22]:
y_prob = RL_scaled.predict_proba(X_test_scaled)[:,1]
umbral = 0.45 # ↓ bajarlo aumenta detección de "No Pagado"
y_pred_umbral = (y_prob >= umbral).astype(int)
print(f"\nReporte de clasificación con umbral ={umbral}")
print(classification_report(y_test, y_pred_umbral, target_names=["No Pagado","Pagado"]))
print("\nMatriz de Confusión con umbral de decisión a0.45:")
print(confusion_matrix(y_test, y_pred_umbral))


Reporte de clasificación con umbral =0.45
              precision    recall  f1-score   support

   No Pagado       0.24      0.50      0.32      1177
      Pagado       0.89      0.72      0.80      6787

    accuracy                           0.69      7964
   macro avg       0.56      0.61      0.56      7964
weighted avg       0.80      0.69      0.73      7964


Matriz de Confusión con umbral de decisión a0.45:
[[ 585  592]
 [1882 4905]]


In [23]:
y_prob = RL_scaled.predict_proba(X_test_scaled)[:, 1]
umbral = 0.40  # ↓ bajarlo aumenta detección de "No Pagado"
y_pred_umbral = (y_prob >= umbral).astype(int)
print(f"\nReporte de clasificación con umbral = {umbral}")
print(classification_report(y_test, y_pred_umbral,
                            target_names=["No Pagado", "Pagado"]))
print("\nMatriz de Confusión con umbral de decisión a 0.40:")
print(confusion_matrix(y_test, y_pred_umbral))


Reporte de clasificación con umbral = 0.4
              precision    recall  f1-score   support

   No Pagado       0.26      0.37      0.30      1177
      Pagado       0.88      0.82      0.85      6787

    accuracy                           0.75      7964
   macro avg       0.57      0.59      0.57      7964
weighted avg       0.79      0.75      0.77      7964


Matriz de Confusión con umbral de decisión a 0.40:
[[ 431  746]
 [1251 5536]]


In [24]:
y_prob = RL_scaled.predict_proba(X_test_scaled)[:,1]
umbral =0.55# ↓ bajarlo aumenta detección de "No Pagado"
y_pred_umbral = (y_prob >= umbral).astype(int)
print(f"\nReporte de clasificación con umbral ={umbral}")
print(classification_report(y_test, y_pred_umbral, target_names=["No Pagado","Pagado"]))
print("\nMatriz de Confusión con umbral de decisión a0.55")
print(confusion_matrix(y_test, y_pred_umbral))


Reporte de clasificación con umbral =0.55
              precision    recall  f1-score   support

   No Pagado       0.21      0.76      0.33      1177
      Pagado       0.92      0.50      0.65      6787

    accuracy                           0.54      7964
   macro avg       0.57      0.63      0.49      7964
weighted avg       0.82      0.54      0.60      7964


Matriz de Confusión con umbral de decisión a0.55
[[ 890  287]
 [3390 3397]]


In [25]:
y_prob = RL_scaled.predict_proba(X_test_scaled)[:,1]
umbral = 0.60  # ↓ bajarlo aumenta detección de "No Pagado"
y_pred_umbral = (y_prob >= umbral).astype(int)
print(f"\nReporte de clasificación con umbral = {umbral}")
print(classification_report(y_test, y_pred_umbral,
                            target_names=["No Pagado","Pagado"]))
print("\nMatriz de Confusión con umbral de decisión a 0.60")
print(confusion_matrix(y_test, y_pred_umbral))


Reporte de clasificación con umbral = 0.6
              precision    recall  f1-score   support

   No Pagado       0.19      0.84      0.31      1177
      Pagado       0.93      0.38      0.54      6787

    accuracy                           0.45      7964
   macro avg       0.56      0.61      0.43      7964
weighted avg       0.82      0.45      0.51      7964


Matriz de Confusión con umbral de decisión a 0.60
[[ 988  189]
 [4205 2582]]


In [26]:
yhat_predmanual = clf2.predict( [[5000,15.5,5,3,4,4,30000,12,12,0]] )

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning:

X does not have valid feature names, but LogisticRegression was fitted with feature names

